Reinforcement Learning with Tunable Qiskit Black-Box

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
if (root / 'qcgpt').exists():
    sys.path.insert(0, str(root))
elif (root.parent / 'qcgpt').exists():
    sys.path.insert(0, str(root.parent))


In [ ]:
import os
import numpy as np
import torch
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity
try:
    from qiskit_aer import AerSimulator
    from qiskit.providers.aer.noise import NoiseModel, depolarizing_error
    AER_AVAILABLE = True
except Exception:
    AER_AVAILABLE = False
from qcgpt.models.policy import CircuitPolicy
from qcgpt.gates import VOCAB, PAD_ID, BOS_CIRC_ID, EOS_CIRC_ID
from qcgpt.data.specs import build_spec_sequence_batch
from qcgpt.training.rollouts import build_batch_specs, RewardBaseline
from qcgpt.encoding import tokens_to_circuit
from qcgpt.simulators.qiskit_sim import circuit_to_qiskit


Configure Qiskit Scoring Black-Box

In [ ]:
use_noise = True
method = 'density_matrix'  # 'statevector' or 'density_matrix'
p1 = 0.001  # 1q depolarizing
p2 = 0.005  # 2q depolarizing
def build_noise_model():
    if not AER_AVAILABLE or not use_noise:
        return None
    nm = NoiseModel()
    nm.add_quantum_error(depolarizing_error(p1, 1), ['x','y','z','h','s','t'])
    nm.add_quantum_error(depolarizing_error(p2, 2), ['cx','cz','swap'])
    return nm
def make_backend(n_qubits):
    nm = build_noise_model()
    if AER_AVAILABLE:
        backend = AerSimulator(method=method, noise_model=nm) if nm else AerSimulator(method=method)
        return backend
    return None
def simulate_output(circ, psi_in):
    qc = QuantumCircuit(circ.nqubits)
    qc.initialize(psi_in, list(range(circ.nqubits)))
    qc2 = circuit_to_qiskit(circ)
    qc.compose(qc2, inplace=True)
    backend = make_backend(circ.nqubits)
    if backend is None or method == 'statevector':
        return Statevector(psi_in).evolve(qc)
    if method == 'density_matrix':
        try:
            qc.save_density_matrix()
            result = backend.run(qc).result()
            rho = result.data(0)['density_matrix']
            return DensityMatrix(rho)
        except Exception:
            return Statevector(psi_in).evolve(qc)
    qc.save_statevector()
    result = backend.run(qc).result()
    sv = result.data(0)['statevector']
    return Statevector(sv)
def compute_reward(spec_tensor, circ, lambda_len=0.1):
    n_states = spec_tensor.shape[0]
    fids = []
    for i in range(n_states):
        rin = spec_tensor[i,0,:,0]
        iin = spec_tensor[i,0,:,1]
        rout = spec_tensor[i,1,:,0]
        iout = spec_tensor[i,1,:,1]
        psi_in = rin.astype(np.float32) + 1j*iin.astype(np.float32)
        psi_out_target = rout.astype(np.float32) + 1j*iout.astype(np.float32)
        pred = simulate_output(circ, psi_in)
        fids.append(state_fidelity(pred, Statevector(psi_out_target)))
    F = float(np.mean(fids))
    return F - lambda_len * len(circ.gates)


Run RL Loop

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CircuitPolicy(vocab_size=len(VOCAB)).to(device)
ckpt_paths = [
    'checkpoints/supervised_best.pt',
    'checkpoints/supervised_final.pt',
    'checkpoints/supervised_current.pt',
]
for p in ckpt_paths:
    if os.path.exists(p):
        state = torch.load(p, map_location=device)
        model.load_state_dict(state['model_state_dict'])
        break
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
baseline = RewardBaseline(momentum=0.9)
num_steps = 500
batch_size = 8
max_len = 32
lambda_len = 0.1
log_every = 50
for step in range(1, num_steps+1):
        spec_states_batch, spec_batch, spec_pad_mask = build_batch_specs(batch_size=batch_size, max_gates_ref=6)
        spec_batch = spec_batch.to(device)
        spec_pad_mask = spec_pad_mask.to(device)
        sampled_tokens, log_probs = model.sample_circuit_tokens(spec_batch, spec_pad_mask, BOS_CIRC_ID, EOS_CIRC_ID, max_len=max_len)
        B, L = sampled_tokens.shape
        rewards = []
        for i in range(B):
            seq = [t for t in sampled_tokens[i].tolist() if t != PAD_ID]
            circ = tokens_to_circuit(seq)
            R = compute_reward(spec_states_batch[i], circ, lambda_len=lambda_len)
            rewards.append(R)
        rewards_t = torch.tensor(rewards, dtype=torch.float32, device=device)
        adv = rewards_t - baseline.value
        loss = -(adv * log_probs).mean()
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        baseline.update(float(rewards_t.mean().item()))
        if step % log_every == 0:
            print(f'[RL] Step {step:04d}  MeanR={rewards_t.mean().item():.4f}  Baseline={baseline.value:.4f}  Loss={loss.item():.4f}')
torch.save({'model_state_dict': model.state_dict()}, 'checkpoints/rl_finetuned.pt')
print('Saved RL-finetuned model to checkpoints/rl_finetuned.pt')
